# Deep Learning Midterm Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

Author: Thomas Kong

NetId: tk2558

Goal: Generate Inferences from Model

## Referenced Data and Docs

### Dataset resources
- Provided train.csv

### Qwen 2B fine-tuning references
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

### **Section 0: Installing Necessary Packages**

> Make sure all packages and their versions are available and correct

> Uncomment whatever is needed to install for notebook environment

> Make sure train.csv and test.csv is uploaded for notebook to access

In [ ]:
# Uncomment the following in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers==4.56.2 accelerate peft bitsandbytes pandas lxml ftfy svgpathtools

# Install Node.js (if not already available)
# !apt-get update -y
# !apt-get install -y nodejs npm

# Install SVGO globally
!npm install -g svgo

In [ ]:
import unsloth, transformers, trl
# CHECK AVAILABLE AND CORRECT VERSIONS
print(transformers.__version__)
print(trl.__version__)
print(unsloth.__version__)

In [ ]:
# Install Node.js (if not already available)
!node -v # Verify Node
!svgo --version # Verify SVGO installation

### **Section 1: Configuration**

> Initialize variables for the notebook and models

> After running all cellblocks in Section 1, you can skip to 2B if you are already using pre-installed training_compressed.csv or skip to Section 7 if you are using pretrained fine-tuned model provided.

In [ ]:
import os
import re
import time
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset, Dataset
import hashlib, random, numpy as np, torch

NETID = "tk2558"
SEED  = int(hashlib.sha256(NETID.encode()).hexdigest(), 16) % 10000
print(f"NetID: {NETID}  |  Seed: {SEED}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
# "unsloth/Qwen3.5-2B-Instruct-bnb-4bit", (Does Not Exist)
# "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit" (VL Model, Attempted)
# "unsloth/Qwen2.5-3B-Instruct-bnb-4bit" (Attempted)
# unsloth/Qwen3-4B-Base-unsloth-bnb-4bit (Attempted)
# "Qwen/Qwen2.5-Coder-3B-Instruct" (Attempted)
# "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit" (Current)

CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 64,
    "learning_rate": 2e-4, #2e-4,
    "num_train_epochs": 1, #1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.05,
    "warmup_steps": 200, #10,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 30000, # Adjust as necessary for GPU 
    "eval_size": 0.02,
    "output_dir": "/kaggle/working/qwen2b_svg_lora",
    #"output_dir": "/content/working/qwen2b_svg_lora",
}

CONFIG

### **Section 3 (IMPORTANT): Secret Key for Hugging Face API**

> Make sure notebook can access the secret key/token and you can assign it to an environment variable.

> If using Kaggle, uncomment top half of codeblock and comment the bottom. If using Google Colab, uncomment bottom half of codeblock and comment the top.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()
# Set the HF_TOKEN environment variable
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

# --------------------------------------------------------------------- #

# from google.colab import userdata
# import os
# # Access the secret and assign it to an environment variable
# # Replace 'MY_API_KEY' with the exact name you used in the Secrets Manager
# api_key = userdata.get("HF_TOKEN")
# os.environ["HF_TOKEN"] = api_key

print("KEY READY")

### **Section 7: Unloading Model**

> You can unload your trained model from where you saved it. This is helpful for reusing trained model for the future.

> Skip to this part and uncomment the codeblock below if you are using pretrained model provided

In [ ]:
# UNLOADING PRE-SAVED MODEL FROM FILE LOCATION

from unsloth import FastLanguageModel
from transformers import AutoModel, AutoTokenizer

MODEL_PATH = "tk2558/qwen_lora_midterm" # Load Model from Hugging Face Example PATH

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModel.from_pretrained(MODEL_PATH, dtype="auto")

### **Section 8 (Inference Code): Generating Output Preparation**

> Initialize functions to prepare for model to start generating outputs. Make sure that the model's outputs are VALID and have a fallback SVG just in case

> (Also make sure model is using GPU)

In [ ]:
SVG_REGEX = re.compile(r"<svg.*?</svg>", re.IGNORECASE | re.DOTALL)

def clean_output(text):
    if "assistant" in text:
        text = text.split("assistant")[-1]
    return text.strip()

def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg_text):
    if not svg_text:
        #print("Invalid: No SVG Text")
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        print("Invalid: Parse Error")
        return False

def extract_svg_features(text):
    color = "black" # DEFAULT
    shape = "circle" # DEFAULT

    color_match = re.search(r'#(?:[0-9a-fA-F]{3}){1,2}', text)
    if color_match:
        color = color_match.group(0)

    if "<rect" in text:
        shape = "rect"
    elif "<ellipse" in text:
        shape = "ellipse"

    return color, shape

def fallback_svg(prompt, failed_output=""):
    color, shape = extract_svg_features(failed_output)
    base = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
    bg = '<rect x="0" y="0" width="256" height="256" fill="white"/>'

    if shape == "rect":
        obj = f'<rect x="64" y="64" width="128" height="128" fill="{color}"/>'

    elif shape == "ellipse":
        obj = f'<ellipse cx="128" cy="128" rx="80" ry="50" fill="{color}"/>'

    else: # shape == path, circle or something else
        obj = f'<circle cx="128" cy="128" r="64" fill="{color}"/>'

    return base + bg + obj + '</svg>'

In [ ]:
def clean_test_prompt(prompt: str) -> str:
    # 1. Fix encoding garbage (ÃƒÂ‚Ã‚Â etc.)
    try:
        import ftfy
        prompt = ftfy.fix_text(prompt)
    except ImportError: # fallback without ftfy — catches the most common double-encoding
        try:
            prompt = prompt.encode('latin-1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            pass

    # 2. Remove meta-instructions that leaked into prompts
    meta_patterns = [
        r"don'?t use markdown[^.]*\.?",
        r"just give svg code[^.]*\.?",
    ]
    for pattern in meta_patterns:
        prompt = re.sub(pattern, '', prompt, flags=re.IGNORECASE)

    # 3. Remove leftover punctuation/whitespace from deletions
    prompt = re.sub(r'\s+', ' ', prompt).strip().strip('.,;:')
    #print(prompt)
    return prompt

In [ ]:
print(torch.cuda.is_available()) # CHECK GPU AVAILABLE
print(model.device) # CHECK GPU IN USE BY MODEL

In [ ]:
model.eval() # SET MODEL TO EVALUATION MODE
FastLanguageModel.for_inference(model) # FAST INFERENCE MODE

### **Section 9 (Inference Code): Generating Outputs**

> Time to Generate Outputs!

> If you want to generate output one prompt at a time, use generate_svg. You can uncomment out text_streamer and streamer=text_streamer if you want to see the output generate in real time.

> If you want to generate outputs from a batch of prompts, use generate_batch_svg.

> We set do_sample=false for Greedy Generation faster outputs

In [ ]:
from transformers import TextStreamer

SYSTEM_PROMPT = (
    "You are an SVG code generator. "
    "When given a description, output ONLY a single valid SVG with these rules:\n"
    "1. Fill the full viewBox — shapes should be large and centered, not small or tucked into a corner.\n"
    "2. Use solid fills and simple strokes. No masks, filters, or external references.\n"
    "3. End output with </svg>.\n"
)

def generate_svg(prompt, max_new_tokens=1024):
    input_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{clean_test_prompt(prompt)}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    text_streamer = TextStreamer(tokenizer, skip_prompt = True)

    #with torch.no_grad():
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False, # GREEDY
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            streamer = text_streamer,
            repetition_penalty=1.1,
        )

    generated_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    decoded = clean_output(decoded)
    svg = extract_svg(decoded)

    if not is_valid_svg(svg):
        #print("Invalid")
        svg = fallback_svg(prompt, decoded)

    return svg

test_time = time.time()
test_prompt = "A stylized red icon depicting a document with a pencil beside it."
pred_svg = generate_svg(test_prompt)
elapsed_time = (time.time() - test_time) # Test Time it takes for One Generation
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))
print(elapsed_time)
print(f"{len(pred_svg)/elapsed_time:.1f} tok/s")

In [ ]:
def generate_batch_svg(prompts, max_new_tokens=2048):
    input_texts = [
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{clean_test_prompt(prompt)}<|im_end|>\n"
        "<|im_start|>assistant\n"
        for prompt in prompts
    ]

    inputs = tokenizer(
        text=input_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CONFIG["max_seq_length"]
    ).to(model.device)

    #text_streamer = TextStreamer(tokenizer, skip_prompt = True)

    with torch.inference_mode():  # upgraded from no_grad
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.1,
            #streamer = text_streamer,
        )

    batch_svgs = []
    input_length = inputs.input_ids.shape[1]

    for i, prompt in enumerate(prompts):
        generated_tokens = output_ids[i][input_length:]

        # Strip padding tokens from the end
        non_pad = (generated_tokens != tokenizer.pad_token_id).nonzero()
        if len(non_pad) > 0:
            generated_tokens = generated_tokens[:non_pad[-1].item() + 1]

        decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        decoded = clean_output(decoded)
        svg = extract_svg(decoded)

        if not is_valid_svg(svg):
            svg = fallback_svg(prompt, decoded)

        batch_svgs.append(svg)

    return batch_svgs

# --- Test ---
DEBUG_PROMPTS = [
    "firewood stack cut logs wood with leaf illustration.",
    "The image shows five horizontal lines of varying thicknesses and lengths, arranged vertically on a white background.",
    "A stylized icon depicting a curved arrow within a square shape. Don't use markdown just give svg code",
    "The image contains black geometric shapes against a white background, forming an abstract representation of a person sitting on a chair.",
    "The image shows a single dark gray triangle pointing upwards, centered against a plain white background.",
]



# Warmup — first generation is always slower due to CUDA kernel init
print("Warming up...")
_ = generate_batch_svg(DEBUG_PROMPTS[:1])

# Test batch sizes
for batch_size in [5]:
    prompts = DEBUG_PROMPTS[:batch_size]
    t0 = time.time()
    results = generate_batch_svg(prompts)
    elapsed = time.time() - t0

    valid = sum(1 for svg in results if is_valid_svg(svg))
    print(f"Batch {batch_size:2d} | {elapsed:.1f}s total | {elapsed/batch_size:.1f}s per SVG | {valid}/{batch_size}")

### **Section 10 (Inference Code): Testing Prompts**

> Test model against the 1000 prompts in test.csv and generate a csv submission for Kaggle

> Two Options:
> *  Use First Codeblock for testing prompts one at a time
> *  Use Second Codeblock for testing prompts in batches

> If submission.csv already exists and accessible, it will pick up from where it left off

> (Make sure clean_training_prompt enable)

In [ ]:
# # Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
# TEST_PROMPTS_PATH = "/kaggle/input/datasets/tk2558/train-prompt/train.csv"
# SUBMISSION_PATH = "/kaggle/working/submission.csv"

# test_df = pd.read_csv(TEST_PROMPTS_PATH)

# total = len(test_df)
# checkpoint = total // 10  # every 10%
# SAVE_EVERY = 10  # save every N samples

# rows = []
# invalid_count = 0
# t0 = time.time()

# # If resuming, load existing file
# if os.path.exists(SUBMISSION_PATH):
#     existing_df = pd.read_csv(SUBMISSION_PATH)
#     rows = existing_df.to_dict("records")
#     start_idx = len(rows)
#     print(f"Resuming from {start_idx}...")
# else:
#     print(f"Starting from the first...")
#     start_idx = 0

# for _, row in test_df.iterrows():
#     svg = generate_svg(row["prompt"])
#     if not is_valid_svg(svg):
#         invalid_count += 1
#         svg = fallback_svg(row["prompt"])

#     rows.append({"id": row["id"], "svg": svg})

#     if (_ + 1) % checkpoint == 0:
#         percent = int(((_ + 1) / total) * 100)
#         print(f"{percent}% done ({_+1}/{total})")

#     if (_ + 1) % SAVE_EVERY == 0:
#         print("Saving")
#         pd.DataFrame(rows).to_csv(SUBMISSION_PATH, index=False)

# sub_df = pd.DataFrame(rows)
# sub_df.to_csv(SUBMISSION_PATH, index=False)

# elapsed_min = (time.time() - t0) / 60
# print(f"Saved: {SUBMISSION_PATH}")
# print(f"Rows: {len(sub_df)}")
# print(f"Invalid/fallback count: {invalid_count}")
# print(f"Invalid SVGs: {invalid_count}/{total}")
# print(f"Runtime (minutes): {elapsed_min:.2f}")
# sub_df.head()

In [ ]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
# BATCH VERSION

TEST_PROMPTS_PATH = "/kaggle/input/datasets/tk2558/train-prompt/train.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

test_df = pd.read_csv(TEST_PROMPTS_PATH)

total = len(test_df)
BATCH_SIZE = 5
SAVE_EVERY = 10  # in batches

rows = []
t0 = time.time()

if os.path.exists(SUBMISSION_PATH):
    existing_df = pd.read_csv(SUBMISSION_PATH)
    rows = existing_df.to_dict("records")
    start_idx = len(rows)
    print(f"Resuming from {start_idx}...")
else:
    print("Starting from the first...")
    start_idx = 0

# Slice off already-processed rows
remaining_df = test_df.iloc[start_idx:].reset_index(drop=True)

prompts = remaining_df["prompt"].tolist()
ids     = remaining_df["id"].tolist()

for batch_num, batch_start in enumerate(range(0, len(prompts), BATCH_SIZE)):
    batch_prompts = prompts[batch_start : batch_start + BATCH_SIZE]
    batch_ids     = ids[batch_start : batch_start + BATCH_SIZE]

    batch_svgs = generate_batch_svg(batch_prompts)
    print("Batch Completed")
    for id_, svg in zip(batch_ids, batch_svgs):
        rows.append({"id": id_, "svg": svg})

    if (batch_num + 1) % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(SUBMISSION_PATH, index=False)
        done = start_idx + batch_start + len(batch_prompts)
        print(f"{done}/{total} done")

# Final save
sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()